# Analisis de columnas del bronze layer

Este notebook analiza en detalle cada una de las 33 columnas seleccionadas
para el bronze layer de loans.

El objetivo es validar con datos reales:
- Que la columna existe en el CSV.
- Que tipo de dato tiene (numerico, string, con simbolos, etc).
- Que valores toma (rango, distribucion, categorias).
- Cuantos nulls tiene y que representan.
- Que edge cases hay (formatos raros, valores fuera de rango, etc).

Con esto validamos o ajustamos el schema definido en `src/lendflow/schemas/loans.py`
antes de correr el job de bronze.

In [1]:
# Setup: SparkSession y lectura del CSV con headers.
# Leemos SIN schema forzado para que Spark tome los tipos y los datos por nombre.

from pyspark.sql import functions as F

from lendflow.utils.spark_session import get_spark

spark = get_spark("column_analysis")

df = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("quote", '"')
    .option("escape", '"')
    .csv("../data/raw/accepted_2007_to_2018Q4.csv.gz")
)

print(f"Filas: {df.count():,}")
print(f"Columnas totales en el CSV: {len(df.columns)}")

26/07/31 08:16:48 WARN Utils: Your hostname, LP-CARLOSLEGUIZAMON resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/31 08:16:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/31 08:16:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/31 08:16:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Filas: 2,260,701
Columnas totales en el CSV: 151


## 1. Columna `id`

El identificador unico del prestamo. Segun mi hipotesis original, viene como
string aunque parezca numero. Vamos a validar:
- Que no tenga nulls (deberia ser 0%).
- Que sea unico por fila (2.26M valores distintos).
- Que valores toma (muestrar 5).
- Si son numericos puros o tienen letras.

In [2]:
# Analisis de la columna 'id'

# Muestra de 5 valores
print("Muestra de valores:")
df.select("id").show(5, truncate=False)

# Cantidad de nulls
nulls = df.filter(F.col("id").isNull()).count()
print(f"Nulls: {nulls:,}")

# Cantidad de valores unicos
distinct = df.select("id").distinct().count()
print(f"Valores unicos: {distinct:,}")

# Chequeamos si TODOS son numericos puros (solo digitos)
non_numeric = df.filter(~F.col("id").rlike("^[0-9]+$") & F.col("id").isNotNull()).count()
print(f"Valores no numericos: {non_numeric:,}")

Muestra de valores:
+--------+
|id      |
+--------+
|68407277|
|68355089|
|68341763|
|66310712|
|68476807|
+--------+
only showing top 5 rows



Nulls: 0


Valores unicos: 2,260,701


Valores no numericos: 33


In [3]:
# Miramos los 33 valores no numericos de id para entender que son
df.filter(
    ~F.col("id").rlike("^[0-9]+$") & F.col("id").isNotNull()
).select("id").show(33, truncate=False)

+------------------------------------------------+
|id                                              |
+------------------------------------------------+
|Total amount funded in policy code 1: 6417608175|
|Total amount funded in policy code 2: 1944088810|
|Total amount funded in policy code 1: 1741781700|
|Total amount funded in policy code 2: 564202131 |
|Total amount funded in policy code 1: 1791201400|
|Total amount funded in policy code 2: 651669342 |
|Total amount funded in policy code 1: 1443412975|
|Total amount funded in policy code 2: 511988838 |
|Total amount funded in policy code 1: 2063142975|
|Total amount funded in policy code 2: 823319310 |
|Total amount funded in policy code 1: 1538432075|
|Total amount funded in policy code 2: 608903141 |
|Total amount funded in policy code 1: 2087217200|
|Total amount funded in policy code 2: 662815446 |
|Total amount funded in policy code 1: 3503840175|
|Total amount funded in policy code 2: 873652739 |
|Total amount funded in policy 

In [4]:
# Vemos como se ven las filas basura en otras columnas
df.filter(
    ~F.col("id").rlike("^[0-9]+$") & F.col("id").isNotNull()
).select("id", "loan_amnt", "issue_d", "loan_status", "grade").show(10, truncate=False)

+------------------------------------------------+---------+-------+-----------+-----+
|id                                              |loan_amnt|issue_d|loan_status|grade|
+------------------------------------------------+---------+-------+-----------+-----+
|Total amount funded in policy code 1: 6417608175|NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 2: 1944088810|NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 1: 1741781700|NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 2: 564202131 |NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 1: 1791201400|NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 2: 651669342 |NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 1: 1443412975|NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 2: 511988838 |NULL     |NULL   |NULL       |NULL |
|Total amount funded in policy code 1: 2063

## 2. Columna `issue_d` (fecha de originacion)

Esta es una columna critica: define el "cohort" del prestamo y la usamos
para calcular vintage curves. Segun mi hipotesis viene como string en
formato "Dec-2018".

Vamos a validar:
- Formato real de los valores.
- Cantidad de nulls (deberian ser bajos, no tiene sentido un prestamo
  sin fecha de originacion).
- Rango de fechas (deberia ir de 2007 a 2018).
- Si hay valores con formato inesperado.

In [5]:
# Analisis de issue_d
# IMPORTANTE: filtramos las filas basura (id no numerico) para no contaminar el analisis

df_clean = df.filter(F.col("id").rlike("^[0-9]+$"))

# Muestra de valores
print("Muestra de valores:")
df_clean.select("issue_d").show(10, truncate=False)

# Nulls
nulls = df_clean.filter(F.col("issue_d").isNull()).count()
print(f"Nulls: {nulls:,}")

# Valores unicos (deberian ser meses distintos)
distinct = df_clean.select("issue_d").distinct().count()
print(f"Valores unicos: {distinct:,}")

# Top 10 valores mas frecuentes
print("\nTop 10 valores mas frecuentes:")
df_clean.groupBy("issue_d").count().orderBy(F.col("count").desc()).show(10, truncate=False)

Muestra de valores:
+--------+
|issue_d |
+--------+
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
|Dec-2015|
+--------+
only showing top 10 rows



Nulls: 0


Valores unicos: 139

Top 10 valores mas frecuentes:


+--------+-----+
|issue_d |count|
+--------+-----+
|Mar-2016|61992|
|Oct-2015|48631|
|May-2018|46311|
|Oct-2018|46305|
|Aug-2018|46079|
|Jul-2015|45962|
|Dec-2015|44343|
|Aug-2017|43573|
|Jul-2018|43089|
|Apr-2018|42928|
+--------+-----+
only showing top 10 rows



## 3. Columna `loan_status`

La columna mas importante del proyecto. Define el estado del prestamo:
current, charged off, late, etc. TODAS las metricas de negocio dependen
de esta columna (vintage curves, roll rates, recovery rate, PAR).

Vamos a validar:
- Cuales son los estados posibles y su distribucion.
- Que no haya valores inesperados o typos.
- Cuantos nulls hay (deberian ser 0).

In [6]:
# Analisis de loan_status

# Nulls
nulls = df_clean.filter(F.col("loan_status").isNull()).count()
print(f"Nulls: {nulls:,}")

# Todos los valores unicos con su count
print("\nDistribucion de loan_status:")
df_clean.groupBy("loan_status").count().orderBy(F.col("count").desc()).show(truncate=False)

Nulls: 0

Distribucion de loan_status:


+---------------------------------------------------+-------+
|loan_status                                        |count  |
+---------------------------------------------------+-------+
|Fully Paid                                         |1076751|
|Current                                            |878317 |
|Charged Off                                        |268559 |
|Late (31-120 days)                                 |21467  |
|In Grace Period                                    |8436   |
|Late (16-30 days)                                  |4349   |
|Does not meet the credit policy. Status:Fully Paid |1988   |
|Does not meet the credit policy. Status:Charged Off|761    |
|Default                                            |40     |
+---------------------------------------------------+-------+



## 4. Columnas con simbolos: `int_rate` y `revol_util`

Segun mi hipotesis original:
- `int_rate` viene como number (double).
- `revol_util` viene con "%" al final.

Vamos a validar ambos.

In [7]:
# Analisis de int_rate y revol_util

print("=== int_rate ===")
df_clean.select("int_rate").show(5, truncate=False)
nulls_int = df_clean.filter(F.col("int_rate").isNull()).count()
print(f"Nulls int_rate: {nulls_int:,}")

# Chequeamos si tiene simbolo %
with_pct_int = df_clean.filter(F.col("int_rate").contains("%")).count()
print(f"Valores con '%' en int_rate: {with_pct_int:,}")

print("\n=== revol_util ===")
df_clean.select("revol_util").show(5, truncate=False)
nulls_rev = df_clean.filter(F.col("revol_util").isNull()).count()
print(f"Nulls revol_util: {nulls_rev:,}")

with_pct_rev = df_clean.filter(F.col("revol_util").contains("%")).count()
print(f"Valores con '%' en revol_util: {with_pct_rev:,}")

=== int_rate ===
+--------+
|int_rate|
+--------+
|13.99   |
|11.99   |
|10.78   |
|14.85   |
|22.45   |
+--------+
only showing top 5 rows



Nulls int_rate: 0


Valores con '%' en int_rate: 0

=== revol_util ===
+----------+
|revol_util|
+----------+
|29.7      |
|19.2      |
|56.2      |
|11.6      |
|64.5      |
+----------+
only showing top 5 rows



Nulls revol_util: 1,802


Valores con '%' en revol_util: 0


## 5. Columnas con formato tipo texto: `term` y `emp_length`

Segun mi hipotesis:
- `term` viene como "36 months" o "60 months".
- `emp_length` viene como "< 1 year", "10+ years", etc.

Vamos a validar.

In [8]:
# Analisis de term y emp_length

print("=== term ===")
df_clean.groupBy("term").count().orderBy(F.col("count").desc()).show(truncate=False)

print("\n=== emp_length ===")
df_clean.groupBy("emp_length").count().orderBy(F.col("count").desc()).show(truncate=False)

=== term ===


+----------+-------+
|term      |count  |
+----------+-------+
| 36 months|1609754|
| 60 months|650914 |
+----------+-------+


=== emp_length ===


+----------+------+
|emp_length|count |
+----------+------+
|10+ years |748005|
|2 years   |203677|
|< 1 year  |189988|
|3 years   |180753|
|1 year    |148403|
|NULL      |146907|
|5 years   |139698|
|4 years   |136605|
|6 years   |102628|
|7 years   |92695 |
|8 years   |91914 |
|9 years   |79395 |
+----------+------+

